# Line Delay Rankings (Polars)

Ranks lines separately by late-only and early-only delay using the Polars analysis path.

In [1]:
from pathlib import Path
import importlib.util
import sys

import plotly.express as px
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "pyproject.toml").exists() and (candidate / "analysis" / "polars").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Run this notebook from the project root or a subdirectory.")

POLARS_ANALYSIS_DIR = PROJECT_ROOT / "analysis" / "polars"
polars_analysis_path = str(POLARS_ANALYSIS_DIR)
if polars_analysis_path in sys.path:
    sys.path.remove(polars_analysis_path)
sys.path.insert(0, polars_analysis_path)

for module_name in ("_shared", "report_cache", "cli_common"):
    module = sys.modules.get(module_name)
    module_file = getattr(module, "__file__", None) if module else None
    module_path = Path(module_file).resolve() if module_file else None
    if module_path is not None and POLARS_ANALYSIS_DIR not in module_path.parents:
        sys.modules.pop(module_name, None)

def load_polars_script(module_name: str, filename: str):
    spec = importlib.util.spec_from_file_location(module_name, POLARS_ANALYSIS_DIR / filename)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module

rankings = load_polars_script("polars_line_delay_rankings", "line-delay-rankings.py")

DB = PROJECT_ROOT / "data" / "foli.db"
CACHE_DIR = PROJECT_ROOT / "outputs" / "polars-report-cache"
FORCE_CACHE = False
NO_CACHE = False
MIN_OBSERVATIONS = 50
LIMIT = 10
TIMEZONE = "Europe/Helsinki"
QUALITY_MODE = "conservative"
BUCKET = "trip-stop"


Set `NO_CACHE = True` to read SQLite directly, or `FORCE_CACHE = True` to rebuild the Polars cache.

In [2]:
class Args:
    db = DB
    cache_dir = CACHE_DIR
    force_cache = FORCE_CACHE
    no_cache = NO_CACHE
    timezone = TIMEZONE
    quality_mode = QUALITY_MODE
    exclude_stop_call_disagreement = False
    bucket = BUCKET
    min_observations = MIN_OBSERVATIONS
    limit = LIMIT

buckets = rankings.load_delay_buckets_for_args(Args)
late = rankings.build_line_rankings(buckets, "late", min_observations=MIN_OBSERVATIONS, limit=LIMIT)
early = rankings.build_line_rankings(buckets, "early", min_observations=MIN_OBSERVATIONS, limit=LIMIT)
late


line_ref,line_name,bucket_count,raw_poll_count,signed_mean_delay_min,median_delay_min,p75_delay_min,p90_delay_min,p95_delay_min,pct_over_3_min_late,pct_over_5_min_late
str,str,u32,i64,f64,f64,f64,f64,f64,f64,f64
"""612""","""612""",1397,3049,4.94,4.12,9.95,15.38,17.52,52.97,46.96
"""615""","""615""",3199,6168,4.39,2.8,9.26,14.79,17.29,48.77,37.95
"""L8""","""L8""",50,1544,3.51,3.66,6.44,11.38,13.53,54.0,36.0
"""614""","""614""",3393,7517,4.43,4.0,6.72,10.23,12.41,59.65,39.91
"""720""","""720""",1601,3357,3.14,3.12,5.58,8.45,9.77,51.28,29.42
"""25""","""25""",13306,25095,2.82,2.06,5.15,8.37,10.85,41.61,26.01
"""V1""","""V1""",2345,7173,2.13,1.45,4.62,8.31,9.37,35.35,22.26
"""42A""","""42A""",1899,4846,2.0,0.87,3.17,8.17,10.17,26.59,18.96
"""24""","""24""",24585,47758,2.35,1.03,4.17,8.07,11.13,32.44,20.39


In [3]:
early


line_ref,line_name,bucket_count,raw_poll_count,signed_mean_delay_min,median_delay_min,pct_early,pct_over_1_min_early,pct_over_3_min_early,median_early_min_abs,p90_early_min_abs
str,str,u32,i64,f64,f64,f64,f64,f64,f64,f64
"""P6""","""P6""",1663,5482,-6.62,-4.23,93.81,83.88,60.85,4.95,16.12
"""903""","""903""",995,3579,0.08,0.0,30.75,24.32,12.66,2.32,12.85
"""901""","""901""",8076,28413,-2.55,0.0,49.24,37.36,24.26,2.92,12.02
"""615""","""615""",3199,6168,4.39,2.8,24.66,18.19,10.03,2.39,9.47
"""N10""","""N10""",1189,5271,-2.69,-2.57,70.14,62.66,46.43,4.7,8.88
"""801""","""801""",45201,104803,-0.52,0.0,48.25,35.3,19.39,2.22,8.57
"""75""","""75""",283,1018,-3.81,-2.55,93.29,85.51,46.64,3.0,8.45
"""L4""","""L4""",1445,5292,-1.92,-1.57,62.84,54.46,34.46,3.53,8.05
"""N7""","""N7""",1593,8216,-1.39,-0.32,53.86,44.7,31.64,3.62,7.96


In [4]:
if late.is_empty():
    print("No matching observations found.")
else:
    fig = px.bar(
        late.sort("p90_delay_min"),
        x="p90_delay_min",
        y="line_ref",
        orientation="h",
        title="Most late lines",
        labels={"line_ref": "Line", "p90_delay_min": "P90 delay, minutes"},
    )
    fig.update_layout(showlegend=False)
    fig.show()
